In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# # Load Dataset
df = pd.read_csv("books_cleaned.csv")

In [3]:
df.head()

,Title,Author,Rating,Rating_Count,Review_Count,Pub_Date,Page_Count,Genres,ISBN,Description,Log_Rating_Count,Popularity_Index,Pub_Year,Description_Word_Count,Title_Word_Count
0,Harry Potter and the Deathly Hallows,J.K. Rowling,4.62,4167736,101662,2007-07-21,784,"Fantasy, Fiction, Young Adult, Harry Potter, M...",9.780550e+12,"It's no longer safe for Harry at Hogwarts, so ...",15.242884,70.422123,2007.0,321,6
1,The Hunger Games,Suzanne Collins,4.35,10088121,269185,2008-09-14,374,"Young Adult, Dystopia, Fiction, Fantasy, Scien...",9.780440e+12,Winning means fame and fortune. Losing means c...,16.126869,70.151881,2008.0,151,3
2,The Kite Runner,Khaled Hosseini,4.36,3548170,116730,2003-05-29,371,"Fiction, Historical Fiction, Classics, Contemp...",9.781590e+12,1970s Twelve-year-old Amir is desperate to win...,15.081943,65.757271,2003.0,81,3
3,The Book Thief,Markus Zusak,4.39,2927965,163552,2005-09-01,592,"Historical Fiction, Fiction, Young Adult, Clas...",NaN,Librarian's note: An alternate cover edition c...,14.889819,65.366303,2005.0,168,3
4,Harry Potter and the Half-Blood Prince,J.K. Rowling,4.58,3714108,72614,2005-07-16,672,"Fantasy, Fiction, Young Adult, Harry Potter, M...",NaN,The war against Voldemort is not going well: e...,15.127649,69.284634,2005.0,192,6


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3999 entries, 0 to 3998
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Title                   3999 non-null   str    
 1   Author                  3999 non-null   str    
 2   Rating                  3999 non-null   float64
 3   Rating_Count            3999 non-null   int64  
 4   Review_Count            3999 non-null   int64  
 5   Pub_Date                3999 non-null   str    
 6   Page_Count              3999 non-null   int64  
 7   Genres                  3999 non-null   str    
 8   ISBN                    3508 non-null   float64
 9   Description             3999 non-null   str    
 10  Log_Rating_Count        3999 non-null   float64
 11  Popularity_Index        3999 non-null   float64
 12  Pub_Year                3999 non-null   float64
 13  Description_Word_Count  3999 non-null   int64  
 14  Title_Word_Count        3999 non-null   int64  
dty

In [5]:
df.isnull().sum()

Title                       0
Author                      0
Rating                      0
Rating_Count                0
Review_Count                0
Pub_Date                    0
Page_Count                  0
Genres                      0
ISBN                      491
Description                 0
Log_Rating_Count            0
Popularity_Index            0
Pub_Year                    0
Description_Word_Count      0
Title_Word_Count            0
dtype: int64

In [6]:
# Drop Unwanted Columns
drop_cols = [
    "ISBN", 
    "Popularity_Index",
    "Title"
]

df.drop(columns=drop_cols, inplace=True)

#isbn->idnetifier only
#Popularity_Index->derived from target
#title->high-cardinality raw text

In [7]:
df.head()

,Author,Rating,Rating_Count,Review_Count,Pub_Date,Page_Count,Genres,Description,Log_Rating_Count,Pub_Year,Description_Word_Count,Title_Word_Count
0,J.K. Rowling,4.62,4167736,101662,2007-07-21,784,"Fantasy, Fiction, Young Adult, Harry Potter, M...","It's no longer safe for Harry at Hogwarts, so ...",15.242884,2007.0,321,6
1,Suzanne Collins,4.35,10088121,269185,2008-09-14,374,"Young Adult, Dystopia, Fiction, Fantasy, Scien...",Winning means fame and fortune. Losing means c...,16.126869,2008.0,151,3
2,Khaled Hosseini,4.36,3548170,116730,2003-05-29,371,"Fiction, Historical Fiction, Classics, Contemp...",1970s Twelve-year-old Amir is desperate to win...,15.081943,2003.0,81,3
3,Markus Zusak,4.39,2927965,163552,2005-09-01,592,"Historical Fiction, Fiction, Young Adult, Clas...",Librarian's note: An alternate cover edition c...,14.889819,2005.0,168,3
4,J.K. Rowling,4.58,3714108,72614,2005-07-16,672,"Fantasy, Fiction, Young Adult, Harry Potter, M...",The war against Voldemort is not going well: e...,15.127649,2005.0,192,6


#### Genre Preprocessing and Encoding

The Genres column contains multiple genre labels stored within a single record. To make this information suitable for machine learning, the genre tags were cleaned, separated, and transformed into encoded numerical features.


In [8]:
df["Genres"].head()

0    Fantasy, Fiction, Young Adult, Harry Potter, M...
1    Young Adult, Dystopia, Fiction, Fantasy, Scien...
2    Fiction, Historical Fiction, Classics, Contemp...
3    Historical Fiction, Fiction, Young Adult, Clas...
4    Fantasy, Fiction, Young Adult, Harry Potter, M...
Name: Genres, dtype: str

In [9]:
print(df["Genres"].iloc[0])
print(type(df["Genres"].iloc[0]))

Fantasy, Fiction, Young Adult, Harry Potter, Magic
<class 'str'>


In [10]:
# convert into list
df["Genres"] = df["Genres"].apply(
    lambda x: [genre.strip().lower() for genre in x.split(",")]
)

Each genre string was split into a list of individual genre labels and standardized by stripping whitespace and converting to lowercase. This transformation allows multi-label encoding of genres.

In [11]:
print(df["Genres"].iloc[0])
print(type(df["Genres"].iloc[0]))

['fantasy', 'fiction', 'young adult', 'harry potter', 'magic']
<class 'list'>


In [12]:
from collections import Counter

all_genres = []

for genres in df["Genres"]:
    all_genres.extend(genres)

genre_counts = Counter(all_genres)

top_genres = pd.DataFrame(
    genre_counts.most_common(20),
    columns=["Genre", "Count"]
)

top_genres

,Genre,Count
0,fiction,2319
1,fantasy,1193
2,romance,820
3,young adult,797
4,historical fiction,666
5,nonfiction,628
6,contemporary,560
7,book club,542
8,mystery,520
9,audiobook,494


A frequency analysis was performed using a Counter to identify the most common genres in the dataset. This helped in selecting meaningful and high-frequency genres for modeling while removing noise such as overly specific or irrelevant tags.

In [13]:
selected_genres = [
    "fiction",
    "fantasy",
    "romance",
    "young adult",
    "historical fiction",
    "nonfiction",
    "mystery",
    "paranormal",
    "science fiction",
    "thriller",
    "horror",
    "biography"
]

In [14]:
mlb = MultiLabelBinarizer()

genre_encoded = pd.DataFrame(
    mlb.fit_transform(df["Genres"]),
    columns=mlb.classes_,
    index=df.index
)

The cleaned genre lists were transformed into numerical features using MultiLabelBinarizer. This technique converts each genre into a binary column indicating its presence (1) or absence (0) for each book.

In [15]:
genre_encoded = genre_encoded[selected_genres]

In [16]:
print(genre_encoded.shape)
print(genre_encoded.columns)

(3999, 12)
Index(['fiction', 'fantasy', 'romance', 'young adult', 'historical fiction',
       'nonfiction', 'mystery', 'paranormal', 'science fiction', 'thriller',
       'horror', 'biography'],
      dtype='str')


In [17]:
genre_encoded.head()

,fiction,fantasy,romance,young adult,historical fiction,nonfiction,mystery,paranormal,science fiction,thriller,horror,biography
0,1,1,0,1,0,0,0,0,0,0,0,0
1,1,1,0,1,0,0,0,0,1,0,0,0
2,1,0,0,0,1,0,0,0,0,0,0,0
3,1,0,0,1,1,0,0,0,0,0,0,0
4,1,1,0,1,0,0,0,0,0,0,0,0


In [18]:
df_ml = pd.concat([df, genre_encoded], axis=1)

In [19]:
df_ml.head()

,Author,Rating,Rating_Count,Review_Count,Pub_Date,Page_Count,Genres,Description,Log_Rating_Count,Pub_Year,...,romance,young adult,historical fiction,nonfiction,mystery,paranormal,science fiction,thriller,horror,biography
0,J.K. Rowling,4.62,4167736,101662,2007-07-21,784,"[fantasy, fiction, young adult, harry potter, ...","It's no longer safe for Harry at Hogwarts, so ...",15.242884,2007.0,...,0,1,0,0,0,0,0,0,0,0
1,Suzanne Collins,4.35,10088121,269185,2008-09-14,374,"[young adult, dystopia, fiction, fantasy, scie...",Winning means fame and fortune. Losing means c...,16.126869,2008.0,...,0,1,0,0,0,0,1,0,0,0
2,Khaled Hosseini,4.36,3548170,116730,2003-05-29,371,"[fiction, historical fiction, classics, contem...",1970s Twelve-year-old Amir is desperate to win...,15.081943,2003.0,...,0,0,1,0,0,0,0,0,0,0
3,Markus Zusak,4.39,2927965,163552,2005-09-01,592,"[historical fiction, fiction, young adult, cla...",Librarian's note: An alternate cover edition c...,14.889819,2005.0,...,0,1,1,0,0,0,0,0,0,0
4,J.K. Rowling,4.58,3714108,72614,2005-07-16,672,"[fantasy, fiction, young adult, harry potter, ...",The war against Voldemort is not going well: e...,15.127649,2005.0,...,0,1,0,0,0,0,0,0,0,0




After feature engineering, the dataset contains numerical features, encoded genre variables, and the target variable in log-transformed form. This ensures the dataset is ready for further preprocessing and feature engineering steps.

In [22]:
# Final target variable check
df_ml["Log_Rating_Count"] = np.log1p(df_ml["Rating_Count"])

#### Target Variable Transformation

The target variable Rating_Count is highly skewed. A log transformation was applied to stabilize variance and improve model performance.

In [23]:
# Remove leakage and unnecessary raw columns
df_ml.drop(columns=["Rating_Count"], inplace=True)
df_ml.drop(columns=["Genres"], inplace=True)



Columns such as Rating_Count were removed to prevent data leakage since the model predicts its log-transformed version. Raw categorical columns like Genres were also removed after encoding.

#### Text Feature Engineering using TF-IDF

The Description column is unstructured text and is converted into numerical features using TF-IDF to extract meaningful word importance while reducing noise from common words.

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=100
)

tfidf_matrix = tfidf.fit_transform(df_ml["Description"])

In [26]:
# Convert TF-IDF to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=df_ml.index
)

In [27]:
# Merge TF-IDF features
df_ml = pd.concat([df_ml, tfidf_df], axis=1)

In [28]:
# Drop raw text column
df_ml.drop(columns=["Description"], inplace=True)

In [30]:
df_ml.head()

,Author,Rating,Review_Count,Pub_Date,Page_Count,Log_Rating_Count,Pub_Year,Description_Word_Count,Title_Word_Count,fiction,...,war,way,woman,women,work,world,year,years,york,young
0,J.K. Rowling,4.62,101662,2007-07-21,784,15.242884,2007.0,321,6,1,...,0.000000,0.0,0.0,0.0,0.0,0.275052,0.130848,0.0,0.0,0.0
1,Suzanne Collins,4.35,269185,2008-09-14,374,16.126869,2008.0,151,3,1,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.191784,0.0,0.0,0.0
2,Khaled Hosseini,4.36,116730,2003-05-29,371,15.081943,2003.0,81,3,1,...,0.000000,0.0,0.0,0.0,0.0,0.214386,0.305964,0.0,0.0,0.0
3,Markus Zusak,4.39,163552,2005-09-01,592,14.889819,2005.0,168,3,1,...,0.000000,0.0,0.0,0.0,0.0,0.112105,0.000000,0.0,0.0,0.0
4,J.K. Rowling,4.58,72614,2005-07-16,672,15.127649,2005.0,192,6,1,...,0.250937,0.0,0.0,0.0,0.0,0.000000,0.237788,0.0,0.0,0.0


In [31]:
df_ml.drop(columns=["Author"], inplace=True)
df_ml.drop(columns=["Pub_Date"], inplace=True)

In [32]:
df_ml.info()

<class 'pandas.DataFrame'>
RangeIndex: 3999 entries, 0 to 3998
Columns: 119 entries, Rating to young
dtypes: float64(103), int64(16)
memory usage: 3.6 MB


In [33]:
df_ml.columns

Index(['Rating', 'Review_Count', 'Page_Count', 'Log_Rating_Count', 'Pub_Year',
       'Description_Word_Count', 'Title_Word_Count', 'fiction', 'fantasy',
       'romance',
       ...
       'war', 'way', 'woman', 'women', 'work', 'world', 'year', 'years',
       'york', 'young'],
      dtype='str', length=119)



The Pub_Date column was removed as it was redundant with Pub_Year. The Author column was excluded due to high cardinality and complexity of encoding. The final dataset consists of numerical, encoded categorical, and TF-IDF features, making it suitable for machine learning models.